# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Source: FlyRank's public report *The State of AI-Driven SEO* (April 2026). It discloses its own standards, among them "a pattern study — found patterns, not proof of cause and effect", and lists ten limitations. The questions below are the ones I'd want asked about my own work, and I answer them for my model in sections 2–4.

### Finding A: the freshness multiplier
*Claim:* among pages older than 365 days, those refreshed in the last 30 days show a 1.6× health score and a 52× impression lift (about 82 → 4.2K impressions) vs. pages last updated 181–360 days ago.

**Where does the label come from?** "Refreshed" is not a measured edit. The paper says refresh detection comes from internal workflow logic and algorithmic signals, and covers anything from a meta tweak to a body rewrite. So the treatment group is "pages the workflow chose to touch".

**Does the design carry the claim?** It's a cross-sectional comparison of two groups, not a before/after on the same pages. My constructive question: **were the refreshed pages already bigger *before* the refresh?** A workflow tends to pick pages that already have impressions to protect. A 52× gap is large enough that selection could explain much of it. A pre-period check (impressions in the 30–60 days *before* each refresh, for both groups) would separate "refresh lifts pages" from "visible pages get refreshed". The paper's "not proof of cause" standard is the right frame; a pre-period row would make the number much stronger.

### Finding B: the growth-prediction model (ML appendix)
*Claim:* 90% accuracy on held-out same-brand pages, 75% on unseen brands, 80/20 split, 20 repeats.

**Where does the label come from?** Growth = more than +10% change over a 90-day window. The question is whether any feature (90-day impressions, position, health score) is computed over the same 90 days as the label. If so, the model partly reads the outcome.

**Does the validation design carry the claim?** Two constructive points:
1. **Accuracy needs its base rate next to it.** With three classes (up / stable / down) and an uneven mix, a model that always says the majority class can already score high. In my starter slice, 54% of pages are "down" and only 15% are "up", so always predicting "not up" would be 85% accurate on an up-vs-rest task. 90% might be a small or a large gain; the report can't tell without the base rate.
2. **The 90% → 75% drop is itself the finding.** The paper did the right thing by reporting both numbers. The 15-point gap says how much of the "same-brand" score was memorizing the brand. For a product used on *new* clients, 75% is the number to quote. The paper already flags the ML pages as "exploratory appendix material", which is exactly the right label.

*(The paper's CTR-by-position-tier finding is close to my lane. I came back to it in Week 2: with coarse tiers, pooled CTR wasn't monotonic in my slice, so I switched to narrow position buckets.)*

In [1]:
import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/content_refresh_anonymized.csv").exists())
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")

# Finding A: can the starter slice support a refreshed-vs-stale comparison?
print("days_since_last_update - top values (share of all pages):")
print(df["days_since_last_update"].value_counts(normalize=True).head(5).round(3).to_string())
old = df[df["content_age_days"] > 365]
print(f"\nPages older than 365 days: {len(old):,}")
print(f"  refreshed <= 30 days ago: {(old['days_since_last_update'] <= 30).sum():,}")
print(f"  last update 181-360 days ago: {old['days_since_last_update'].between(181, 360).sum():,}")
print("-> update ages are heaped on a few values and the 'stale' comparison group is empty here:")
print("   the refresh flag behaves like a workflow event, so a pre-period check is the right test.")

# Finding B: base rates for a 'growth' label in my slice
share = df["trend_direction"].value_counts(normalize=True)
print("\ntrend_direction mix:", share.round(3).to_dict())
print(f"Always predicting 'not up' on up-vs-rest = {1 - share['up']:.1%} accuracy, before any model.")

days_since_last_update - top values (share of all pages):
days_since_last_update
20     0.386
104    0.292
22     0.119
8      0.064
13     0.017

Pages older than 365 days: 6,360
  refreshed <= 30 days ago: 5,807
  last update 181-360 days ago: 0
-> update ages are heaped on a few values and the 'stale' comparison group is empty here:
   the refresh flag behaves like a workflow event, so a pre-period check is the right test.

trend_direction mix: {'down': 0.542, 'stable': 0.199, 'up': 0.146, 'new': 0.075, 'flat': 0.038}
Always predicting 'not up' on up-vs-rest = 85.4% accuracy, before any model.


## 2. My model under an honest split (before/after)

Same Week-5 random forest, same features, same label (`stays_under` in the last 30 days), same metrics. Only the validation design changes:

| Design | What it simulates |
|---|---|
| **Before: random 5-fold** | Pages from one client land in both train and test. Like the paper's "same-brand" number. |
| **After 1: GroupKFold by client** (my Week-5 setup) | Test clients are never seen in training. |
| **After 2: time-forward + grouped** | Train on *days 61–90 → days 31–60*, test on *days 31–60 → last 30 days*, on unseen clients. This is the deployment situation: learn from the past, score the next month for a new client. |

The earliest window comes from `90d − prev_30d − last_30d`. I checked it's never negative, so the three 30-day windows add up. The baseline (frozen Week-4 rule) is scored on the same test rows each time.

In [2]:
import sklearn
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
SEED = 42

for m in ["clicks", "impressions", "sessions"]:
    df[f"{m}_early_30d"] = df[f"{m}_90d"] - df[f"{m}_prev_30d"] - df[f"{m}_last_30d"]
    assert (df[f"{m}_early_30d"] >= 0).all()

base = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 1000)
          & (df["impressions_prev_30d"] >= 100) & (df["impressions_last_30d"] >= 100)].copy()
base["bucket"] = pd.cut(base["avg_position"], [0, 3, 5, 10, 15, 20], labels=["1-3", "3-5", "5-10", "10-15", "15-20"])
base = base.reset_index(drop=True)

def build(d, fw, lw):
    """Features from window fw, label from the later window lw. Same recipe as Week 5."""
    d = d.copy()
    for w in (fw, lw):
        d[f"ctr_{w}"] = 100 * d[f"clicks_{w}_30d"] / d[f"impressions_{w}_30d"]
        t = d.groupby("bucket", observed=True)[[f"clicks_{w}_30d", f"impressions_{w}_30d"]].sum()
        d[f"exp_{w}"] = d["bucket"].map(100 * t[f"clicks_{w}_30d"] / t[f"impressions_{w}_30d"]).astype(float)
    X = pd.DataFrame({
        "log_imp": np.log1p(d[f"impressions_{fw}_30d"]), "log_clk": np.log1p(d[f"clicks_{fw}_30d"]),
        "ctr": d[f"ctr_{fw}"], "ratio": d[f"ctr_{fw}"] / d[f"exp_{fw}"], "avg_position": d["avg_position"],
        "log_sess": np.log1p(d[f"sessions_{fw}_30d"]), "content_age_days": d["content_age_days"],
        "has_word_count": d["word_count"].notna().astype(int), "word_count": d["word_count"].fillna(0)})
    X = pd.concat([X, pd.get_dummies(d[["main_intent", "content_type"]], dummy_na=True, dtype=int)], axis=1)
    y = (d[f"ctr_{lw}"] < 0.5 * d[f"exp_{lw}"]).astype(int)
    potential = (d[f"exp_{fw}"] / 100 * d[f"impressions_{fw}_30d"] - d[f"clicks_{fw}_30d"]).clip(lower=0)
    rule = np.where(d[f"ctr_{fw}"] < 0.5 * d[f"exp_{fw}"], potential, 0.0)
    return X, y, rule

p_at = lambda y, s, k: y[np.argsort(-s, kind="mergesort")[:k]].mean()
make_rf = lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=SEED, n_jobs=-1)

def evaluate(name, X_tr_all, y_tr_all, X_te_all, y_te_all, rule, splits):
    rows = []
    for tr, te in splits:
        s = make_rf().fit(X_tr_all.iloc[tr], y_tr_all.iloc[tr]).predict_proba(X_te_all.iloc[te])[:, 1]
        y = y_te_all.values[te]
        rows.append({"model_auc": roc_auc_score(y, s), "model_p@50": p_at(y, s, 50),
                     "rule_auc": roc_auc_score(y, rule[te]), "rule_p@50": p_at(y, rule[te], 50), "base_rate": y.mean()})
    r = pd.DataFrame(rows)
    out = r.mean().to_dict()
    out["model_p@50 range"] = f"{r['model_p@50'].min():.2f}-{r['model_p@50'].max():.2f}"
    return pd.Series(out, name=name)

X, y, rule = build(base, "prev", "last")
results = [
    evaluate("Before: random 5-fold", X, y, X, y, rule, list(KFold(5, shuffle=True, random_state=SEED).split(X))),
    evaluate("After 1: grouped by client", X, y, X, y, rule, list(GroupKFold(5).split(X, y, base["client_id"]))),
]
# Time-forward: needs measurable CTR in the earliest window too
tf = base[base["impressions_early_30d"] >= 100].reset_index(drop=True)
X_past, y_past, _ = build(tf, "early", "prev")
X_next, y_next, rule_next = build(tf, "prev", "last")
X_past, X_next = X_past.align(X_next, join="outer", axis=1, fill_value=0)
results.append(evaluate("After 2: time-forward + grouped", X_past, y_past, X_next, y_next, rule_next,
                        list(GroupKFold(5).split(X_next, y_next, tf["client_id"]))))
before_after = pd.DataFrame(results)
print(f"Pages: {len(base):,} (time-forward subset: {len(tf):,}), clients: {base['client_id'].nunique()}")
print(f"Label base rate: {y.mean():.1%} | in the earlier (training) period of the time-forward design: {y_past.mean():.1%}")
before_after.round(3)

Pages: 9,664 (time-forward subset: 9,625), clients: 28
Label base rate: 37.8% | in the earlier (training) period of the time-forward design: 42.6%


,model_auc,model_p@50,rule_auc,rule_p@50,base_rate,model_p@50 range
Before: random 5-fold,0.807,0.884,0.713,0.804,0.378,0.82-0.92
After 1: grouped by client,0.791,0.804,0.703,0.696,0.355,0.62-0.94
After 2: time-forward + grouped,0.779,0.744,0.702,0.696,0.355,0.60-0.94


**What the before/after shows:**
- **The random split looks better, but mostly not because of memorization.** Model precision@50 drops from about 0.88 (random) to about 0.80 (grouped). The untrained rule drops just as much (about 0.80 → 0.70), though, and the rule can't memorize anything. So most of that change comes from how pages spread across folds (one client holds about 39% of pages), not from the model learning clients. ROC-AUC barely moves (0.81 → 0.79), and the model-minus-rule gap is about the same in both rows. What the grouped split *does* change is the spread: p@50 ranges 0.62–0.94 instead of 0.82–0.92. In my slice, client memorization is measured as small, unlike the paper's 15-point same-brand gap. A likely reason is that my features are CTR *relative to position*, not brand-specific levels.
- **Time-forward is the strictest test and the model still measures above the rule.** Trained only on the earlier month pair and scored on the next month for unseen clients, the forest measured about 0.74 precision@50 vs about 0.70 for the frozen rule, and ROC-AUC about 0.78 vs about 0.70. The gap to the rule shrinks from about 11 points (grouped) to about 5 points. **Time-forward + grouped is the number I'll quote from now on.**
- The label base rate also moved between periods (about 43% vs 38%). The share of under-capturing pages isn't stable month to month, which is one more reason to judge the ranking by precision against the base rate, not by raw counts.

## 3. Leakage audit

Same hunt as Week 3, now on my final feature set. Three checks:
1. **Timeline:** every feature must be knowable before the label window starts.
2. **Harness test:** deliberately add a leaky column and confirm the score jumps. If it didn't, my test harness would be broken.
3. **Suspect removal:** train without the one feature I know overlaps the label window (`avg_position`, a 90-day average) and see whether the result depends on it.

Then real failure examples from the honest (grouped) run.

In [3]:
timeline = pd.DataFrame([
    ("log_imp / log_clk / ctr / ratio / log_sess", "days 31-60", "clean: strictly before the label window"),
    ("content_age_days, word_count, intent, type", "static", "clean: page context, not outcome"),
    ("avg_position", "90-day average", "OVERLAP: includes the label window - tested below"),
    ("trend_pct / trend_direction", "90-day trend", "excluded: computed over the label window"),
    ("*_last_30d, *_90d totals", "contain last 30 days", "excluded: contain the label window"),
    ("days_since_last_update", "snapshot at export", "excluded: an update inside the label window shows up"),
    ("client_id / content_id", "-", "excluded: grouping only"),
], columns=["feature", "window", "verdict"])
print(timeline.to_string(index=False))

groups = list(GroupKFold(5).split(X, y, base["client_id"]))
checks = {
    "Honest feature set": X,
    "+ LEAK: ctr in label window": X.assign(leak=100 * base["clicks_last_30d"] / base["impressions_last_30d"]),
    "+ LEAK: clicks_90d (contains label window)": X.assign(leak=np.log1p(base["clicks_90d"])),
    "+ trend_pct (overlapping product field)": X.assign(leak=base["trend_pct"]),
    "- avg_position (remove known overlap)": X.drop(columns=["avg_position"]),
}
audit = pd.DataFrame([evaluate(k, v, y, v, y, rule, groups) for k, v in checks.items()])[["model_auc", "model_p@50", "model_p@50 range", "base_rate"]]
print("\nLeakage harness - grouped by client, same folds:")
audit.round(3)

                                   feature               window                                              verdict
log_imp / log_clk / ctr / ratio / log_sess           days 31-60              clean: strictly before the label window
content_age_days, word_count, intent, type               static                     clean: page context, not outcome
                              avg_position       90-day average    OVERLAP: includes the label window - tested below
               trend_pct / trend_direction         90-day trend             excluded: computed over the label window
                  *_last_30d, *_90d totals contain last 30 days                   excluded: contain the label window
                    days_since_last_update   snapshot at export excluded: an update inside the label window shows up
                    client_id / content_id                    -                              excluded: grouping only



Leakage harness - grouped by client, same folds:


,model_auc,model_p@50,model_p@50 range,base_rate
Honest feature set,0.791,0.804,0.62-0.94,0.355
+ LEAK: ctr in label window,0.997,1.000,1.00-1.00,0.355
+ LEAK: clicks_90d (contains label window),0.860,1.000,1.00-1.00,0.355
+ trend_pct (overlapping product field),0.796,0.792,0.58-0.96,0.355
- avg_position (remove known overlap),0.788,0.800,0.56-1.00,0.355


**Reading the audit:**
- **The harness works.** Adding the label-window CTR sends ROC-AUC to about 1.0 and precision@50 to 1.0. Adding 90-day clicks pushes precision@50 to 1.0 as well. If my honest number looked like that, it would be a confession. It doesn't (about 0.79 AUC).
- **`trend_pct` adds almost nothing** for this label, but it's still excluded on timeline grounds: it is computed over the window I'm predicting.
- **Removing `avg_position` barely moves the score** (mean AUC about 0.79 either way; the fold range widens). So the known 90-day overlap is not what drives the result. I keep it out of the claims anyway and list it as a limitation.

Now the failures: what the honest model gets wrong.

In [4]:
# Out-of-fold scores from the grouped run, then the model's most confident picks
oof = np.zeros(len(X))
for tr, te in groups:
    oof[te] = make_rf().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
e = base.assign(score=oof, y=y.values, imp_change=base["impressions_last_30d"] / base["impressions_prev_30d"])
top = e[e["score"] >= np.quantile(oof, 0.9)].copy()
top["outcome"] = np.where(top["y"] == 1, "true positive (stayed under)", "false positive (recovered)")
print(f"Top 10% picks: {len(top):,} | false-positive rate {(top['y'] == 0).mean():.1%} vs base rate {y.mean():.1%}")
print("\nMedians by outcome - impressions in the feature month and how impressions changed next month:")
print(top.groupby("outcome")[["impressions_prev_30d", "clicks_prev_30d", "imp_change"]].median().round(2).to_string())

miss = e[(e["y"] == 1) & (e["score"] < 0.3)]
print(f"\nMissed pages (stayed under, score < 0.3): {len(miss):,}; median impression change {miss['imp_change'].median():.2f}x "
      f"vs {e['imp_change'].median():.2f}x for all pages")

examples = pd.concat([
    top[top["y"] == 0].sort_values("score", ascending=False).head(3).assign(case="false positive"),
    miss.sort_values("impressions_last_30d", ascending=False).head(2).assign(case="false negative"),
])
print("\nFive failure examples (no IDs):")
examples[["case", "score", "avg_position", "impressions_prev_30d", "clicks_prev_30d",
          "impressions_last_30d", "clicks_last_30d", "imp_change"]].round(3).reset_index(drop=True)

Top 10% picks: 967 | false-positive rate 23.0% vs base rate 37.8%

Medians by outcome - impressions in the feature month and how impressions changed next month:
                              impressions_prev_30d  clicks_prev_30d  imp_change
outcome                                                                        
false positive (recovered)                   951.5              0.0        0.64
true positive (stayed under)                1601.0              0.0        0.68

Missed pages (stayed under, score < 0.3): 440; median impression change 0.78x vs 0.73x for all pages

Five failure examples (no IDs):


,case,score,avg_position,impressions_prev_30d,clicks_prev_30d,impressions_last_30d,clicks_last_30d,imp_change
0,false positive,0.792,11.9,7705,1,818,10,0.106
1,false positive,0.792,9.7,9331,2,1313,4,0.141
2,false positive,0.783,8.1,44418,32,12232,27,0.275
3,false negative,0.154,6.4,30526,65,105052,151,3.441
4,false negative,0.267,8.7,28339,54,86232,87,3.043


**What the failures look like:**
- **False positives are thinner pages.** The model's confident wrong picks had a median of about 950 impressions in the feature month vs about 1,600 for correct picks, and both groups had a median of zero clicks. Zero clicks on about a thousand impressions is partly chance, and the model can't tell bad luck from a real gap.
- **The extreme failures share an impression shift.** The three most confident false positives lost 70–90% of their impressions next month while clicks stayed about level or rose. The two biggest misses roughly tripled their impressions while CTR fell. **But as a group the shift is weak:** median impression change is about 0.64× for false positives vs 0.68× for true positives, and 0.78× for missed pages vs 0.73× overall. So "impression-mix shift" explains the worst cases, not the typical error. I'm treating it as a hypothesis.
- The extreme cases point to one missing signal: **query mix** (which queries a page shows for). The warehouse has query-level data; the starter CSV doesn't. That's my next feature to test, not a fix I can claim yet.

## 4. Claim rewrite

I re-read my Week-5 notebook looking for sentences that go further than the evidence. Five of them did, and I've rewritten them **in the Week-5 notebook itself** (markdown only; the numbers are unchanged). The cell below prints each before/after and checks that the strong wording is gone.

My boldest sentence was *"It is clearly better at telling a lasting CTR gap from a wobble."* "Clearly" ignored overlapping fold ranges, and the time-forward test in section 2 shows the gap to the rule is about half as big as the grouped number suggested.

**The claim I'll carry into the capstone:**
> On a 28-client starter slice, a random forest trained on one month and scored on the next month for unseen clients measured precision@50 of about 0.74 for "page still under-captures CTR for its position", vs about 0.70 for a transparent rule and a 38% base rate. This is a directional, decision-support result: it orders an editor's review queue. It does not show that rewriting a title recovers clicks.

In [5]:
REWRITES = [('- **Every model beats the frozen rule on ROC-AUC and precision.** The random forest is best on precision@50, about 0.80 vs about 0.70 for the rule. It is clearly better at telling a lasting CTR gap from a wobble.', '- **On held-out clients, every model measured a higher mean ROC-AUC and precision than the frozen rule.** The random forest had the highest mean precision@50 (about 0.80 vs about 0.70 for the rule), but the per-fold ranges overlap (0.62–0.94 vs 0.54–0.86), so the size of the gain is uncertain.'), ("- **The combined ranking (forest × potential) is the useful result.** It keeps the rule's value focus (clicks-gap captured@50 about 0.41, slightly above the rule) at about the same precision. That's a modest, directional gain, not a breakthrough.", "- **The combined ranking (forest × potential) is the most balanced option observed.** It keeps the rule's value focus (clicks-gap captured@50 about 0.41 vs 0.39 for the rule, within fold noise) at about the same precision. That's a directional result, not a measured improvement."), ("**Recommendation for the queue:** use *forest probability × missed-click potential*, and keep the rule's reason code (`LOW_CTR_FOR_POSITION`) for explainability.", "**Directional suggestion for a decision-support queue:** rank by *forest probability × missed-click potential* and keep the rule's reason code (`LOW_CTR_FOR_POSITION`) for explainability. The editor still decides; the score only orders the review."), ("That is plausible: the best predictor of next month's gap is this month's gap, *measured relative to position*.", "That is plausible: in this slice, the strongest measured signal for next month's gap is this month's gap, *measured relative to position*."), ("Ranking by *probability × missed-click potential* matched the rule's value and improved precision slightly.", "Ranking by *probability × missed-click potential* matched the rule's value at similar precision (differences within fold noise).")]

for i, (before, after) in enumerate(REWRITES, 1):
    print(f"[{i}] BEFORE: {before.lstrip('- ')}")
    print(f"    AFTER:  {after.lstrip('- ')}\n")

import nbformat
w05 = nbformat.read(root / "work/notebooks/w05_model.ipynb", as_version=4)
md = "\n".join(c.source for c in w05.cells if c.cell_type == "markdown")
RISKY = r"\b(clearly|proves?|caus(e|es|ed)|guarantee[sd]?|always|is best)\b"
print("Rewrites present in w05:", all(after in md for _, after in REWRITES))
print("Old sentences left in w05:", sum(before in md for before, _ in REWRITES))
print("Risky words left in w05 markdown:", sorted({m.group(0).lower() for m in re.finditer(RISKY, md, flags=re.I)}) or "none")

[1] BEFORE: **Every model beats the frozen rule on ROC-AUC and precision.** The random forest is best on precision@50, about 0.80 vs about 0.70 for the rule. It is clearly better at telling a lasting CTR gap from a wobble.
    AFTER:  **On held-out clients, every model measured a higher mean ROC-AUC and precision than the frozen rule.** The random forest had the highest mean precision@50 (about 0.80 vs about 0.70 for the rule), but the per-fold ranges overlap (0.62–0.94 vs 0.54–0.86), so the size of the gain is uncertain.

[2] BEFORE: **The combined ranking (forest × potential) is the useful result.** It keeps the rule's value focus (clicks-gap captured@50 about 0.41, slightly above the rule) at about the same precision. That's a modest, directional gain, not a breakthrough.
    AFTER:  **The combined ranking (forest × potential) is the most balanced option observed.** It keeps the rule's value focus (clicks-gap captured@50 about 0.41 vs 0.39 for the rule, within fold noise) at about t

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.